In [1]:
from xarray_utils import analyze_netcdf, zarr_to_netcdf, find_missing_days

In [2]:
import xarray as xr

# Open the Zarr dataset
ds_imd = xr.open_zarr("IMD_rainfall_0p25.zarr")
ds_imd = ds_imd.where(ds_imd != -999)

# Save as NetCDF
# ds_imd.to_netcdf("IMD_rainfall_0p25.nc")

In [40]:
import xarray as xr

# Open the Zarr dataset
ds_ecm = xr.open_zarr("s2s_reforecast_sorted.zarr")

# Save as NetCDF
ds_ecm.to_netcdf("s2s_reforecast.nc")

In [4]:
analyze_netcdf("IMD_rainfall_0p25.nc")


Analysis for NetCDF File: IMD_rainfall_0p25.nc

--- Dimensions ---
time: 31046
lat: 129
lon: 135

--- Coordinates ---
- lat:
    dtype: float64
    shape: (129,)
    attributes: {'axis': 'Y', 'long_name': 'latitude', 'standard_name': 'latitude', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (135,)
    attributes: {'axis': 'X', 'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- time:
    dtype: datetime64[ns]
    shape: (31046,)
    attributes: {'long_name': 'time', 'standard_name': 'time'}

--- Data Variables ---
- rain:
    dtype: float64
    shape: (31046, 129, 135)
    dimensions: ('time', 'lat', 'lon')
    attributes: {'long_name': 'Rainfall', 'units': 'mm/day'}

--- Global Attributes ---
Conventions: CF-1.7
comment: 
crs: epsg:4326
history: 2026-06-19 06:45:25.930728 Python
references: 
source: https://imdpune.gov.in/
title: IMD gridded data



In [41]:
analyze_netcdf("s2s_reforecast.nc")

Analysis for NetCDF File: s2s_reforecast.nc

--- Dimensions ---
time: 3720
step: 43
lat: 33
lon: 35

--- Coordinates ---
- lat:
    dtype: float64
    shape: (33,)
    attributes: {'long_name': 'latitude', 'standard_name': 'latitude', 'stored_direction': 'decreasing', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (35,)
    attributes: {'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- step:
    dtype: int64
    shape: (43,)
- time:
    dtype: datetime64[ns]
    shape: (3720,)
    attributes: {'long_name': 'initial time of forecast', 'standard_name': 'forecast_reference_time'}

--- Data Variables ---
- 10m_u_component_of_wind:
    dtype: float32
    shape: (3720, 43, 33, 35)
    dimensions: ('time', 'step', 'lat', 'lon')
    attributes: {'GRIB_NV': np.int64(0), 'GRIB_Nx': np.int64(35), 'GRIB_Ny': np.int64(33), 'GRIB_cfName': 'eastward_wind', 'GRIB_cfVarName': 'u10', 'GRIB_dataType': 'cf', 'GRIB_gridDefinitionDescription': 'Latitude/longi

In [ ]:
attrs_df = pd.DataFrame(
    ds_ecm["total_precipitation"].attrs.items(),
    columns=["Attribute", "Value"]
)

print(attrs_df.to_string(index=False))

NameError: name 'pd' is not defined

In [9]:
import pandas as pd
import numpy as np
import xarray as xr

# convert lead days into timedeltas
step_td = pd.to_timedelta(ds_ecm.step.values, unit="D").to_numpy()

ds_ecmv = ds_ecm.assign_coords(
    valid_time=(("time", "step"),
                ds_ecm.time.values[:, None] + step_td[None, :])
)


In [10]:
print(ds_ecmv.coords)

Coordinates:
  * time        (time) datetime64[ns] 30kB 2005-01-01 2005-01-03 ... 2024-12-31
  * step        (step) int64 344B 0 1 2 3 4 5 6 7 8 ... 35 36 37 38 39 40 41 42
    valid_time  (time, step) datetime64[ns] 1MB 2005-01-01 ... 2025-02-11
  * lat         (lat) float64 264B 38.5 37.5 36.5 35.5 34.5 ... 9.5 8.5 7.5 6.5
  * lon         (lon) float64 280B 66.5 67.5 68.5 69.5 ... 97.5 98.5 99.5 100.5


In [11]:
import pandas as pd

df = pd.DataFrame({
    "Lead": ds_ecmv.step.values,
    "Valid Date": ds_ecmv.valid_time.isel(time=2).dt.strftime("%Y-%m-%d").values
})

print(df)

    Lead  Valid Date
0      0  2005-01-05
1      1  2005-01-06
2      2  2005-01-07
3      3  2005-01-08
4      4  2005-01-09
5      5  2005-01-10
6      6  2005-01-11
7      7  2005-01-12
8      8  2005-01-13
9      9  2005-01-14
10    10  2005-01-15
11    11  2005-01-16
12    12  2005-01-17
13    13  2005-01-18
14    14  2005-01-19
15    15  2005-01-20
16    16  2005-01-21
17    17  2005-01-22
18    18  2005-01-23
19    19  2005-01-24
20    20  2005-01-25
21    21  2005-01-26
22    22  2005-01-27
23    23  2005-01-28
24    24  2005-01-29
25    25  2005-01-30
26    26  2005-01-31
27    27  2005-02-01
28    28  2005-02-02
29    29  2005-02-03
30    30  2005-02-04
31    31  2005-02-05
32    32  2005-02-06
33    33  2005-02-07
34    34  2005-02-08
35    35  2005-02-09
36    36  2005-02-10
37    37  2005-02-11
38    38  2005-02-12
39    39  2005-02-13
40    40  2005-02-14
41    41  2005-02-15
42    42  2005-02-16


In [12]:
import numpy as np
import pandas as pd

target_date = np.datetime64("2005-01-03")

mask = ds_ecmv.valid_time == target_date

time_idx, step_idx = np.where(mask)

rows = []

for t, s in zip(time_idx, step_idx):
    rows.append({
        "Initialization": str(ds_ecmv.time.values[t])[:10],
        "Lead": int(ds_ecmv.step.values[s]),
        "Valid Date": str(ds_ecmv.valid_time.values[t, s])[:10]
    })

df = pd.DataFrame(rows)

print(df)

  Initialization  Lead  Valid Date
0     2005-01-01     2  2005-01-03
1     2005-01-03     0  2005-01-03


In [13]:
missing_days = find_missing_days("s2s_reforecast.nc")

Found 3585 missing day(s) between 2005-01-01 and 2024-12-31.
First 10 Missing days (or all if < 10):
  - 2005-01-02
  - 2005-01-04
  - 2005-01-06
  - 2005-01-08
  - 2005-01-10
  - 2005-01-12
  - 2005-01-14
  - 2005-01-16
  - 2005-01-18
  - 2005-01-20
  ... and 3575 more.


In [14]:
ds_ecmv["total_precipitation"].isel(time=3719)

<xarray.DataArray 'total_precipitation' (step: 43, lat: 33, lon: 35)> Size: 199kB
[49665 values with dtype=float32]
Coordinates:
  * step        (step) int64 344B 0 1 2 3 4 5 6 7 8 ... 35 36 37 38 39 40 41 42
    valid_time  (step) datetime64[ns] 344B 2024-12-31 2025-01-01 ... 2025-02-11
  * lat         (lat) float64 264B 38.5 37.5 36.5 35.5 34.5 ... 9.5 8.5 7.5 6.5
  * lon         (lon) float64 280B 66.5 67.5 68.5 69.5 ... 97.5 98.5 99.5 100.5
    time        datetime64[ns] 8B 2024-12-31
Attributes: (12/31)
    GRIB_NV:                                  0
    GRIB_Nx:                                  35
    GRIB_Ny:                                  33
    GRIB_cfName:                              unknown
    GRIB_cfVarName:                           tp
    GRIB_dataType:                            cf
    ...                                       ...
    GRIB_typeOfLevel:                         surface
    GRIB_units:                               kg m**-2
    GRIB_uvRelativeToGrid:                    0
    long_name:                                Total Precipitation
    standard_name:                            unknown
    units:                                    kg m**-2

In [15]:
print(ds_ecmv["total_precipitation"])

<xarray.DataArray 'total_precipitation' (time: 3720, step: 43, lat: 33, lon: 35)> Size: 739MB
[184753800 values with dtype=float32]
Coordinates:
  * time        (time) datetime64[ns] 30kB 2005-01-01 2005-01-03 ... 2024-12-31
  * step        (step) int64 344B 0 1 2 3 4 5 6 7 8 ... 35 36 37 38 39 40 41 42
    valid_time  (time, step) datetime64[ns] 1MB 2005-01-01 ... 2025-02-11
  * lat         (lat) float64 264B 38.5 37.5 36.5 35.5 34.5 ... 9.5 8.5 7.5 6.5
  * lon         (lon) float64 280B 66.5 67.5 68.5 69.5 ... 97.5 98.5 99.5 100.5
Attributes: (12/31)
    GRIB_NV:                                  0
    GRIB_Nx:                                  35
    GRIB_Ny:                                  33
    GRIB_cfName:                              unknown
    GRIB_cfVarName:                           tp
    GRIB_dataType:                            cf
    ...                                       ...
    GRIB_typeOfLevel:                         surface
    GRIB_units:                        

In [16]:
for var in ds_ecmv.data_vars:
    print(f"{var}: {ds_ecmv[var].isnull().sum().values}")
    print(f"{var} min: {ds_ecmv[var].min().values}")
    print(f"{var} max: {ds_ecmv[var].max().values}")

10m_u_component_of_wind: 0
10m_u_component_of_wind min: -26.513259887695312
10m_u_component_of_wind max: 26.615392684936523
10m_v_component_of_wind: 0
10m_v_component_of_wind min: -24.005157470703125
10m_v_component_of_wind max: 26.9791259765625
2m_temperature: 4296600
2m_temperature min: 228.89710998535156
2m_temperature max: 317.254150390625
mean_sea_level_pressure: 0
mean_sea_level_pressure min: 96751.40625
mean_sea_level_pressure max: 107060.578125
specific_humidity_1000: 0
specific_humidity_1000 min: 2.5238567104679532e-05
specific_humidity_1000 max: 0.02706041932106018
specific_humidity_200: 0
specific_humidity_200 min: 8.861678679750185e-08
specific_humidity_200 max: 0.0004495223402045667
specific_humidity_500: 0
specific_humidity_500 min: 5.677500212186715e-07
specific_humidity_500 max: 0.008800661191344261
specific_humidity_850: 0
specific_humidity_850 min: 1.6343616152880713e-05
specific_humidity_850 max: 0.022664396092295647
total_precipitation: 0
total_precipitation min: -0

In [17]:
ds_imd.coords

Coordinates:
  * time     (time) datetime64[ns] 248kB 1941-01-01 1941-01-02 ... 2025-12-31
  * lat      (lat) float64 1kB 6.5 6.75 7.0 7.25 7.5 ... 37.75 38.0 38.25 38.5
  * lon      (lon) float64 1kB 66.5 66.75 67.0 67.25 ... 99.25 99.5 99.75 100.0

In [18]:
imd_sub = ds_imd.sel(time=slice('2005-01-01', '2025-02-11'))

In [19]:

for date in imd_sub.time.values:

    # Find all ECMWF forecasts valid on this date
    mask = ds_ecmv.valid_time.astype("datetime64[D]") == target_date


In [21]:
ds_ecmv.data_vars

Data variables:
    10m_u_component_of_wind  (time, step, lat, lon) float32 739MB -0.8921 ......
    10m_v_component_of_wind  (time, step, lat, lon) float32 739MB 1.038 ... -...
    2m_temperature           (time, step, lat, lon) float32 739MB nan ... 299.7
    mean_sea_level_pressure  (time, step, lat, lon) float32 739MB 1.02e+05 .....
    specific_humidity_1000   (time, step, lat, lon) float32 739MB 0.002437 .....
    specific_humidity_200    (time, step, lat, lon) float32 739MB 7.966e-06 ....
    specific_humidity_500    (time, step, lat, lon) float32 739MB 0.0006166 ....
    specific_humidity_850    (time, step, lat, lon) float32 739MB 0.001817 .....
    total_precipitation      (time, step, lat, lon) float32 739MB 0.0 ... 7.641

In [22]:
import numpy as np

samples = []

for target_date in imd_sub.time.values:

    mask = ds_ecmv.valid_time.astype("datetime64[D]") == target_date
    time_idx, step_idx = np.where(mask)

    # IMD target
    y = imd_sub["rain"].sel(time=target_date).values

    for t, s in zip(time_idx, step_idx):

        sample = {
            "date": target_date,
            "lead": int(ds_ecmv.step.values[s]),
            "target": y,
            "10m_u_component_of_wind": ds_ecmv["10m_u_component_of_wind"].isel(time=t, step=s).values,
            "10m_v_component_of_wind": ds_ecmv["10m_v_component_of_wind"].isel(time=t, step=s).values,
            "2m_temperature": ds_ecmv["2m_temperature"].isel(time=t, step=s).values,
            "mean_sea_level_pressure": ds_ecmv["mean_sea_level_pressure"].isel(time=t, step=s).values,
            "specific_humidity_1000": ds_ecmv["specific_humidity_1000"].isel(time=t, step=s).values,
            "specific_humidity_200": ds_ecmv["specific_humidity_200"].isel(time=t, step=s).values,
            "specific_humidity_500": ds_ecmv["specific_humidity_500"].isel(time=t, step=s).values,
            "specific_humidity_850": ds_ecmv["specific_humidity_850"].isel(time=t, step=s).values,
            "total_precipitation": ds_ecmv["total_precipitation"].isel(time=t, step=s).values,
        }

        samples.append(sample)

In [ ]:
samples[20]

{'date': np.datetime64('2005-01-09T00:00:00.000000000'),
 'lead': 8,
 'target': array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], shape=(129, 135)),
 '10m_u_component_of_wind': array([[-0.3635559 , -0.4050598 , -0.2661438 , ...,  0.7306824 ,
          0.7909851 ,  0.27902222],
        [-0.6596985 , -0.1621399 , -0.18630981, ...,  0.86642456,
          1.141571  ,  0.44821167],
        [-0.39382935, -0.22927856, -0.38137817, ...,  0.62545776,
          0.68063354,  0.43917847],
        ...,
        [-4.081085  , -4.0051575 , -3.936554  , ..., -3.0810852 ,
         -2.7534485 , -4.365265  ],
        [-4.0188293 , -3.5312805 , -3.936798  , ..., -5.0364075 ,
         -4.0276184 , -4.9062805 ],
        [-4.497589  , -4.46756   , -4.4739075 , ..., -5.712433  ,
  

In [30]:
samples.dtype

AttributeError: 'list' object has no attribute 'dtype'

In [26]:
from sklearn.dummy import DummyRegressor

if len(X) < 2:
    print("Not enough valid samples to train a baseline model.")
else:
    baseline = DummyRegressor(strategy="mean")
    baseline.fit(X_train, y_train)

    baseline_pred = baseline.predict(X_test)
    baseline_r2 = r2_score(y_test, baseline_pred)
    baseline_rmse = root_mean_squared_error(y_test, baseline_pred)

    print(f"Baseline Test R^2: {baseline_r2:.4f}")
    print(f"Baseline Test RMSE: {baseline_rmse:.4f}")


Baseline Test R^2: -0.0015
Baseline Test RMSE: 3.6181


In [27]:
rain_mean = float(ds_imd["rain"].mean(skipna=True).values)
print(f"Mean rainfall: {rain_mean:.4f}")


Mean rainfall: 3.1522


KeyError: 'lat'

In [35]:
y[4]

0.8134201323894197